# Machine Learning — Sprint 3

| US | Descrizione | Responsabile |
|---|---|---|
| US-14 | Preprocessing delle variabili | Youssra |
| US-15 | Split Train/Test e Baseline | Youssra |
| US-16 | SelectKBest e Logistic Regression | Youssra |
| US-17 | Stepwise Forward e Backward | Youssra |
| US-18 | Decision Tree | Youssra |
| US-19 | Random Forest | Youssra |

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/processed/data_clean.csv')
print('Dataset caricato: ' + str(df.shape[0]) + ' righe, ' + str(df.shape[1]) + ' colonne')

---

## US-14 · Preprocessing delle Variabili

**Responsabile**: Youssra Zarouky

### Classificazione delle variabili

| Tipo | Variabili | Trasformazione |
|---|---|---|
| Numeriche continue | voti, eta, tassi economici | StandardScaler |
| Binarie | genere, borsa, debiti... | passthrough (gia 0/1) |
| Categoriche ordinali | titolo studio genitori | OrdinalEncoder |

### Nota sul Data Leakage
Le variabili del 1o e 2o semestre (esami superati, voti) sono incluse perche il dataset rappresenta studenti al termine del percorso. In un sistema di intervento precoce andrebbero escluse, ma per questo esercizio le manteniamo documentando la scelta.

In [ ]:
from src.preprocessing import build_preprocessing_pipeline

preprocessor, numerical_cols, binary_cols, ordinal_cols = build_preprocessing_pipeline()

feature_cols = numerical_cols + binary_cols + ordinal_cols
X = df[feature_cols]
y = df['target']

print('Feature totali: ' + str(len(feature_cols)))
print('Numeriche: ' + str(len(numerical_cols)))
print('Binarie: ' + str(len(binary_cols)))
print('Ordinali: ' + str(len(ordinal_cols)))
print()
print('Distribuzione target:')
print(y.value_counts())

### Decisione
La pipeline e costruita con ColumnTransformer in `src/preprocessing.py` e verra usata in tutte le US successive. Le variabili categoriche nominali con alta cardinalita (course, nationality) sono escluse per evitare esplosione dimensionale con One-Hot Encoding.

---

## US-15 · Split Train/Test e Baseline

**Responsabile**: Youssra Zarouky

Split 80/20 stratificato per preservare la proporzione delle classi. Il DummyClassifier definisce la baseline minima che i modelli reali devono superare.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train set: ' + str(X_train.shape[0]) + ' righe')
print('Test set:  ' + str(X_test.shape[0]) + ' righe')
print()
print('Distribuzione target nel train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Distribuzione target nel test:')
print(y_test.value_counts(normalize=True).round(3))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

dummy_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent', random_state=42))
])

dummy_pipeline.fit(X_train, y_train)
y_pred_dummy = dummy_pipeline.predict(X_test)

print('=== BASELINE — DummyClassifier (most_frequent) ===')
print(classification_report(y_test, y_pred_dummy, target_names=['Graduate', 'Dropout']))

### Risultato Baseline

Il DummyClassifier predice sempre la classe piu frequente (Graduate). Ottiene un'accuracy apparentemente alta ma F1-Score per Dropout = 0 — non identifica nessuno studente a rischio.

Questo conferma che **l'accuracy da sola non e una metrica affidabile** per questo problema. Useremo F1-Score macro e per classe come riferimento per valutare i modelli successivi.

---

## US-16 · SelectKBest e Logistic Regression

**Responsabile**: Youssra Zarouky

Selezione delle 10 variabili piu informative e allenamento di una Logistic Regression come modello interpretabile di riferimento.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Preprocessing manuale per SelectKBest
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

selector = SelectKBest(f_classif, k=10)
selector.fit(X_train_prep, y_train)

# Nomi delle feature dopo il ColumnTransformer
feature_names = numerical_cols + binary_cols + ordinal_cols
scores = pd.DataFrame({'feature': feature_names, 'score': selector.scores_})
scores = scores.sort_values('score', ascending=False)

print('Top 10 feature:')
print(scores.head(10).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top10 = scores.head(10).sort_values('score')
ax.barh(top10['feature'], top10['score'], color='#3498db')
ax.set_xlabel('Score F-classif')
ax.set_title('Top 10 Feature — SelectKBest', fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/us16_selectkbest_top10.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafico salvato.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('selector', SelectKBest(f_classif, k=10)),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print('=== Logistic Regression — Top 10 Feature ===')
print(classification_report(y_test, y_pred_lr, target_names=['Graduate', 'Dropout']))

### Osservazioni

Le variabili semestrali (esami superati e voti) dominano la classifica, confermando quanto emerso nell'analisi esplorativa (US-07). Questo e coerente: chi supera pochi esami tende ad abbandonare.

La Logistic Regression supera nettamente la baseline — in particolare migliora il recall per la classe Dropout, che e la classe di interesse per l'universita.

Essendo un modello lineare, la Logistic Regression ha pero limiti nell'intercettare relazioni non-lineari tra le variabili.

---

## US-17 · Selezione Feature con Stepwise Forward e Backward

**Responsabile**: Youssra Zarouky

Confronto tra tre metodi di selezione feature: SelectKBest, Forward e Backward. Attenzione: questa cella puo impiegare qualche minuto.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
import pandas as pd

estimator = LogisticRegression(class_weight='balanced', max_iter=200, random_state=42)
feature_names = numerical_cols + binary_cols + ordinal_cols

print('Avvio Forward Selection...')
sfs_forward = SequentialFeatureSelector(estimator, n_features_to_select=10, direction='forward', cv=3, n_jobs=-1)
sfs_forward.fit(X_train_prep, y_train)
forward_features = [feature_names[i] for i in range(len(feature_names)) if sfs_forward.get_support()[i]]
print('Forward completato.')

print('Avvio Backward Selection...')
sfs_backward = SequentialFeatureSelector(estimator, n_features_to_select=10, direction='backward', cv=3, n_jobs=-1)
sfs_backward.fit(X_train_prep, y_train)
backward_features = [feature_names[i] for i in range(len(feature_names)) if sfs_backward.get_support()[i]]
print('Backward completato.')

In [ ]:
top10_kbest = list(scores.head(10)['feature'])

comparison = pd.DataFrame({
    'SelectKBest': [f in top10_kbest for f in feature_names],
    'Forward': [f in forward_features for f in feature_names],
    'Backward': [f in backward_features for f in feature_names]
}, index=feature_names)

comparison = comparison[comparison.any(axis=1)]
print('Confronto metodi di selezione feature:')
print(comparison.to_string())

### Osservazioni

I tre metodi tendono a concordare sulle variabili semestrali (esami superati e voti) come le piu informative — coerente con l'analisi esplorativa.

Le differenze tra i metodi riflettono approcci diversi:
- **SelectKBest**: valuta ogni feature indipendentemente
- **Forward**: aggiunge una feature alla volta scegliendo quella che migliora di piu
- **Backward**: parte da tutte le feature e rimuove la meno utile

Le feature selezionate da almeno 2 metodi su 3 sono le piu robuste e saranno prioritarie nei modelli successivi.

---

## US-18 · Decision Tree

**Responsabile**: Youssra Zarouky

Allenamento e ottimizzazione del Decision Tree al variare della profondita massima. La curva train/test mostra il punto di overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

depths = [2, 3, 5, 7, 10, 15]
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, class_weight='balanced', random_state=42)
    dt.fit(X_train_prep, y_train)
    train_scores.append(f1_score(y_train, dt.predict(X_train_prep), average='macro'))
    test_scores.append(f1_score(y_test, dt.predict(X_test_prep), average='macro'))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, marker='o', label='Train', color='#3498db')
ax.plot(depths, test_scores, marker='o', label='Test', color='#e74c3c')
ax.set_xlabel('max_depth')
ax.set_ylabel('F1-Score Macro')
ax.set_title('Decision Tree — Curva Train/Test al variare di max_depth', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/us18_dt_depth_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curva salvata.')

In [ ]:
from sklearn.metrics import classification_report

best_depth = depths[test_scores.index(max(test_scores))]
print('Miglior max_depth: ' + str(best_depth))

best_dt = DecisionTreeClassifier(max_depth=best_depth, class_weight='balanced', random_state=42)
best_dt.fit(X_train_prep, y_train)
y_pred_dt = best_dt.predict(X_test_prep)

print('=== Decision Tree — max_depth=' + str(best_depth) + ' ===')
print(classification_report(y_test, y_pred_dt, target_names=['Graduate', 'Dropout']))

In [ ]:
feature_names = numerical_cols + binary_cols + ordinal_cols
importances = pd.Series(best_dt.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances.index, importances.values, color='#3498db')
ax.set_xlabel('Importanza')
ax.set_title('Feature Importance — Decision Tree', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/us18_dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Feature importance salvata.')

In [ ]:
from sklearn.tree import plot_tree

dt_vis = DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42)
dt_vis.fit(X_train_prep, y_train)

fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(dt_vis, feature_names=feature_names, class_names=['Graduate', 'Dropout'],
    filled=True, rounded=True, fontsize=9, ax=ax)
plt.title('Decision Tree (max_depth=3)', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/us18_dt_tree_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Albero salvato.')

### Osservazioni

La curva train/test mostra chiaramente il punto di overfitting: con depth elevata il modello impara a memoria il training set ma non generalizza al test.

Le feature piu importanti sono le variabili semestrali — in particolare il numero di esami superati e il voto medio — confermando i risultati dell'analisi esplorativa.

Il Decision Tree con max_depth ottimale supera la baseline del DummyClassifier, ma il modello lineare (Logistic Regression) rimane competitivo grazie alla sua regolarizzazione.